# Assignment 12

In [32]:
import pandas as pd
import os
import glob


folders = [
    "../../data/kinect_good_preprocessed_not_cut_A11_mediapipe",
    "../../data/kinect_good_preprocessed_A9_mediapipe"
]

pairs = [
    ('left_shoulder', 'right_shoulder'),
    ('left_elbow', 'right_elbow'),
    ('left_wrist', 'right_wrist'),
    ('left_hip', 'right_hip'),
    ('left_knee', 'right_knee'),
    ('left_ankle', 'right_ankle')
]

for target_folder in folders:
    all_files = glob.glob(os.path.join(target_folder, "*.csv"))
    print(f"Processing folder: {target_folder} ({len(all_files)} files found)")
    
    count = 0
    for f in all_files:
        if "_mirrored" in f: 
            continue
        
        df = pd.read_csv(f)
        mirrored_df = df.copy()
        
        x_cols = [c for c in df.columns if "_3d_x" in c]
        mirrored_df[x_cols] = 1.0 - df[x_cols]
        
        for left, right in pairs:
            for axis in ['x', 'y', 'z']:
                l_col = f"{left}_3d_{axis}"
                r_col = f"{right}_3d_{axis}"
                
                if l_col in df.columns and r_col in df.columns:
                    mirrored_df[l_col] = df[r_col]
                    mirrored_df[r_col] = df[l_col]
                    
                    if axis == 'x':
                        mirrored_df[l_col] = 1.0 - mirrored_df[l_col]
                        mirrored_df[r_col] = 1.0 - mirrored_df[r_col]

        new_path = f.replace(".csv", "_mirrored.csv")
        mirrored_df.to_csv(new_path, index=False)
        count += 1

    print(f"Successfully created {count} mirrored files in {target_folder}.\n")

print("All folders processed.")

Processing folder: ../../data/kinect_good_preprocessed_not_cut_A11_mediapipe (360 files found)
Successfully created 180 mirrored files in ../../data/kinect_good_preprocessed_not_cut_A11_mediapipe.

Processing folder: ../../data/kinect_good_preprocessed_A9_mediapipe (358 files found)
Successfully created 179 mirrored files in ../../data/kinect_good_preprocessed_A9_mediapipe.

All folders processed.


## Data labelling - Creating data
Here we join the uncut and cut files to create the actual data. This creates the files we use for training, validation aswell as testing.

In [33]:
import pandas as pd
import os
import glob
from tqdm import tqdm


cut_folder = "../../data/kinect_good_preprocessed_A9_mediapipe"
uncut_folder = "../../data/kinect_good_preprocessed_not_cut_A11_mediapipe"
output_folder = "../../data/labeled_files" 

os.makedirs(output_folder, exist_ok=True)

uncut_files = glob.glob(os.path.join(uncut_folder, "*.csv"))

print(f"Found {len(uncut_files)} uncut files. Starting labeling...")

for uncut_path in tqdm(uncut_files):
    filename = os.path.basename(uncut_path)
    cut_path = os.path.join(cut_folder, filename)
    
    if not os.path.exists(cut_path):
        print(f"Warning: No cut file found for {filename}. Skipping.")
        continue
        
    df_uncut = pd.read_csv(uncut_path)
    df_cut = pd.read_csv(cut_path)
    

    start_frame = df_cut['FrameNo'].min()
    end_frame = df_cut['FrameNo'].max()
    
    # Label 1 if the frame is within the range, otherwise 0
    df_uncut['activity_label'] = (
        (df_uncut['FrameNo'] >= start_frame) & 
        (df_uncut['FrameNo'] <= end_frame)
    ).astype(int)
    
    save_path = os.path.join(output_folder, filename)
    df_uncut.to_csv(save_path, index=False)

print(f"\nDone! Labeled files saved to: {output_folder}")

Found 360 uncut files. Starting labeling...


 37%|███▋      | 132/360 [00:01<00:02, 109.63it/s]

 97%|█████████▋| 349/360 [00:03<00:00, 113.28it/s]

100%|██████████| 360/360 [00:03<00:00, 109.99it/s]


Done! Labeled files saved to: ../../data/labeled_files


## Training the model

## Configuration

In [34]:
#config = {
#    "epochs": 30,
#    "patience": 5,
#    "batch_size": 64,
#    "lr": 0.0005,
#    "hidden_size": 128,
#    "num_layers": 2,
#    "seq_length": 5,
#    "dropout": 0.2,
#    "input_size": 39,
#    "note": "Baseline"
#}

# List of files that are corrupted or invalid recordings
ignore_list = ["B2_kinect.csv", "B3_kinect.csv", "B4_kinect.csv", "B5_kinect.csv", "A130_kinect.csv"]

experiment_variants = [


    # 3. Balanced GRU (Medium context, robust initialization)
    {
        "note": "GRU_Balanced_Refined",
        "seed": 42,
        "seq_length": 8,          # Mid-range context
        "hidden_size": 128,
        "lr": 0.0008,             # Slightly aggressive learning rate
        "batch_size": 64,
        "epochs": 30,
        "patience": 5,
        "num_layers": 2,
        "dropout": 0.2,
        "input_size": 39,
        "use_scaling": True,
        "activation": "identity",
        "init": "xavier",
        "bidirectional": True,
        "rnn_type": "GRU",
    },
    # 4. Dense Uni-directional LSTM (Testing if look-ahead is necessary)
    {
        "note": "LSTM_Dense_Uni",
        "seed": 42,
        "seq_length": 10,
        "hidden_size": 128,
        "lr": 0.0005,
        "batch_size": 64,
        "epochs": 30,
        "patience": 5,
        "num_layers": 2,
        "dropout": 0.1,
        "input_size": 39,
        "use_scaling": False,     # Testing raw coordinate performance
        "activation": "identity",
        "init": "default",
        "bidirectional": False,   # Faster inference, testing impact on MAE_Stop
        "rnn_type": "LSTM",
    },
    # 5. Lightweight Responsive GRU (Minimalist architecture)
    {
        "note": "GRU_Light_Rapid",
        "seed": 42,
        "seq_length": 7,
        "hidden_size": 64,
        "lr": 0.001,
        "batch_size": 128,        # Large batch for stable gradients in light model
        "epochs": 30,
        "patience": 5,
        "num_layers": 1,
        "dropout": 0.0,
        "input_size": 39,
        "use_scaling": True,
        "activation": "relu",     # Using ReLU for faster activation
        "init": "kaiming",
        "bidirectional": True,
        "rnn_type": "GRU",
    }

]

### Classes

In [35]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
import torch.nn.init as init

class ActivityGatekeeper(nn.Module):
    def __init__(self, config):
        super(ActivityGatekeeper, self).__init__()
        
        # 1. Determine if we are using Bidirectional
        self.bidirectional = config.get("bidirectional", False)
        multiplier = 2 if self.bidirectional else 1
        
        # 2. Select RNN Type (LSTM or GRU)
        rnn_type = config.get("rnn_type", "LSTM")
        
        if rnn_type == "GRU":
            self.rnn = nn.GRU(
                config["input_size"], 
                config["hidden_size"], 
                config["num_layers"], 
                batch_first=True, 
                dropout=config["dropout"] if config["num_layers"] > 1 else 0,
                bidirectional=self.bidirectional
            )
        else:
            self.rnn = nn.LSTM(
                config["input_size"], 
                config["hidden_size"], 
                config["num_layers"], 
                batch_first=True, 
                dropout=config["dropout"] if config["num_layers"] > 1 else 0,
                bidirectional=self.bidirectional
            )
        
        # 3. Activation variant for the hidden state
        if config.get("activation") == "relu":
            self.act = nn.ReLU()
        elif config.get("activation") == "tanh":
            self.act = nn.Tanh()
        else:
            self.act = nn.Identity()
            
        # 4. Final Output Layer
        self.fc = nn.Linear(config["hidden_size"] * multiplier, 1)
        
        # 5. Initialization variant
        if config.get("init") == "xavier":
            init.xavier_uniform_(self.fc.weight)
        elif config.get("init") == "kaiming":
            init.kaiming_normal_(self.fc.weight)

    def forward(self, x):
        out, _ = self.rnn(x)
        last_step = out[:, -1, :]
        activated = self.act(last_step)
        return self.fc(activated)

class ActivityDataset(Dataset):
    def __init__(self, file_paths, config, scaler=None):
        self.X, self.y = [], []
        seq_length = config["seq_length"]
        for path in file_paths:
            df = pd.read_csv(path)
            feat_cols = [c for c in df.columns if any(s in c for s in ['_x','_y','_z']) and 'FrameNo' not in c]
            X_data, y_data = df[feat_cols].values.astype('float32'), df['activity_label'].values.astype('float32')
            if scaler is not None:
                X_data = scaler.transform(X_data)

            if len(df) < seq_length: continue
            for i in range(len(df) - seq_length + 1):
                self.X.append(X_data[i : i + seq_length])
                self.y.append(y_data[i + seq_length - 1])
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return torch.tensor(self.X[idx]), torch.tensor([self.y[idx]])

def get_with_mirrors(file_list):
    res = []
    for f in file_list:
        res.append(f)
        m = f.replace(".csv", "_mirrored.csv")
        if os.path.exists(m): res.append(m)
    return res

import random
import numpy as np
import torch

def set_seed(seed=42):
    # Python's built-in random module
    random.seed(seed)
    
    # Numpy
    np.random.seed(seed)
    
    # PyTorch (CPU and GPU)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    
    # CuDNN determinism (Crucial for LSTMs on GPU)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"✅ Random seed set to: {seed}")




### Resample to make the dataset more balanced

In [36]:
import numpy as np

data_path = "../../data/labeled_files/"
originals = sorted([
    f for f in glob.glob(os.path.join(data_path, "*.csv")) 
    if "_mirrored" not in f and os.path.basename(f) not in ignore_list
])
all_labels = []
for f in originals:
    all_labels.extend(pd.read_csv(f)['activity_label'].values)

neg, pos = np.bincount(np.array(all_labels).astype(int))
pos_weight = torch.tensor([neg / pos], dtype=torch.float).to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Imbalance ratio: {neg/pos:.2f}")

Imbalance ratio: 0.58


### Training and evaluation

In [37]:
import mlflow
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import os
import joblib
import dagshub
import sys
sys.path.append("../../scripts")
import ml_utils as mlutils
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns

# --- Setup ---
dagshub.init(repo_owner="SamuelFredricBerg", repo_name="4dt907", mlflow=True)
project_name = "Start_Stop_Predictor_ModelV2"
utils = mlutils.mlutils(project_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

kf = KFold(n_splits=10, shuffle=True, random_state=42)

for config in experiment_variants:
    set_seed(config["seed"])
    best_config_for_run = config.copy()
    best_mae_for_run = float('inf')
    with mlflow.start_run(run_name="A12 - new metrics") as run:
        mlflow.log_params(config)
        
        # Track frame differences across all files in all folds
        all_start_diffs = []
        all_stop_diffs = []
        all_file_results = [] # List to store dicts for the final CSV report
        best_overall_mae = float('inf')

        for fold, (train_idx, test_idx) in enumerate(kf.split(originals)):
            print(f"\n--- Fold {fold+1} ---")
            train_orig = [originals[i] for i in train_idx]
            test_orig = [originals[i] for i in test_idx]

            # Fit Scaler
            scaler = None
            if config.get("use_scaling", False):
                scaler = MinMaxScaler()
                train_samples = [pd.read_csv(f)[[c for c in pd.read_csv(f).columns if '_3d_' in c]].values for f in train_orig]
                scaler.fit(np.vstack(train_samples))

            # Loaders - IMPORTANT: Pass scaler to ActivityDataset
            train_loader = DataLoader(ActivityDataset(get_with_mirrors(train_orig[:int(len(train_orig)*0.9)]), config, scaler), batch_size=config["batch_size"], shuffle=True)
            val_loader = DataLoader(ActivityDataset(get_with_mirrors(train_orig[int(len(train_orig)*0.9):]), config, scaler), batch_size=config["batch_size"], shuffle=False)

            model = ActivityGatekeeper(config).to(device)
            optimizer = optim.Adam(model.parameters(), lr=config["lr"])
            criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

            best_v_loss, patience_count = float('inf'), 0
            fold_ckpt = f"best_fold_{fold}.pth"

            # Training
            for epoch in range(config["epochs"]):
                model.train()
                for x, y in train_loader:
                    optimizer.zero_grad(); criterion(model(x.to(device)), y.to(device)).backward(); optimizer.step()
                
                model.eval(); v_loss = 0
                with torch.no_grad():
                    for x, y in val_loader: v_loss += criterion(model(x.to(device)), y.to(device)).item()
                
                if (v_loss/len(val_loader)) < best_v_loss:
                    best_v_loss = v_loss/len(val_loader); patience_count = 0
                    torch.save(model.state_dict(), fold_ckpt)
                else:
                    patience_count += 1
                    if patience_count >= config["patience"]: break

            # --- New Metric Evaluation: Boundary MAE ---
            model.load_state_dict(torch.load(fold_ckpt))
            model.eval()
            
            fold_start_errs, fold_stop_errs = [], []

            for test_file in test_orig:
                df = pd.read_csv(test_file)
                # Actual Boundaries
                actual_indices = df[df['activity_label'] == 1]['FrameNo'].values
                if len(actual_indices) == 0: continue
                actual_start, actual_stop = actual_indices[0], actual_indices[-1]

                # Inference
                feat_cols = [c for c in df.columns if any(s in c for s in ['_x','_y','_z']) and 'FrameNo' not in c]
                X_eval = scaler.transform(df[feat_cols].values) if scaler else df[feat_cols].values
                seq_len = config["seq_length"]
                preds = [0] * (seq_len - 1)
                
                with torch.no_grad():
                    for i in range(len(df) - seq_len + 1):
                        window = torch.tensor(X_eval[i:i+seq_len]).float().unsqueeze(0).to(device)
                        preds.append(1 if torch.sigmoid(model(window)).item() >= 0.5 else 0)

                # Predicted Boundaries
                pred_indices = np.where(np.array(preds) == 1)[0]
                if len(pred_indices) > 0:
                    p_start = df.iloc[pred_indices[0]]['FrameNo']
                    p_stop = df.iloc[pred_indices[-1]]['FrameNo']
                    start_off = abs(p_start - actual_start)
                    stop_off = abs(p_stop - actual_stop)
                else:
                    # Penalty for failing to detect any exercise
                    start_off = len(df) 
                    stop_off = len(df)

                total_off = start_off + stop_off
                fold_start_errs.append(start_off)
                fold_stop_errs.append(stop_off)
                all_start_diffs.append(start_off)
                all_stop_diffs.append(stop_off)

                # Store for the report
                all_file_results.append({
                    "filename": os.path.basename(test_file),
                    "start_off": start_off,
                    "stop_off": stop_off,
                    "total_off": total_off
                })

            # Record Fold Results
            if fold_start_errs:
                all_start_diffs.extend(fold_start_errs)
                all_stop_diffs.extend(fold_stop_errs)
                fold_mae = (np.mean(fold_start_errs) + np.mean(fold_stop_errs)) / 2
                
                if fold_mae < best_overall_mae:
                    best_overall_mae = fold_mae
                    best_config_for_run = config.copy()
                    torch.save(model.state_dict(), "gatekeeper_best.pth")
                    joblib.dump(scaler, "scaler_best.joblib")

            if os.path.exists(fold_ckpt): os.remove(fold_ckpt)

        report_df = pd.DataFrame(all_file_results)
        
        # 1. Full detailed report
        report_df.to_csv("full_file_performance.csv", index=False)
        mlflow.log_artifact("full_file_performance.csv")

        # 2. Top 5 Best Predicted (Lowest total_off)
        best_5 = report_df.nsmallest(5, 'total_off')
        best_5.to_csv("top_5_best_files.csv", index=False)
        mlflow.log_artifact("top_5_best_files.csv")

        # 3. Top 5 Worst Predicted (Highest total_off)
        worst_5 = report_df.nlargest(5, 'total_off')
        worst_5.to_csv("top_5_worst_files.csv", index=False)
        mlflow.log_artifact("top_5_worst_files.csv")

        # Summary Metrics
        mae_start = np.mean(all_start_diffs)
        mae_stop = np.mean(all_stop_diffs)
        std_start = np.std(all_start_diffs)
        std_stop = np.std(all_stop_diffs)
        total_std = std_start + std_stop / 2
        total_mae = (mae_start + mae_stop) / 2
        mlflow.log_metric("MAE_Start_Frames", mae_start)
        mlflow.log_metric("MAE_Stop_Frames", mae_stop)
        mlflow.log_metric("MAE_Total_Average", total_mae)

        print(f"\n--- Experiment: {config['note']} ---")
        print(f"Top 5 Worst Performers:\n{worst_5}")
        print(f"Top 5 Best Performers:\n{best_5}")

        print(f"\nFinal MAE Result -> Total: {total_mae:.2f} frames, std: {total_std:.2f}")
        print(f"Start: {mae_start:.2f} (±{std_start:.2f}) | Stop: {mae_stop:.2f} (±{std_stop:.2f})")

        if utils.auto_check_challenger(run.info.run_id, metric_name="MAE_Total_Average"):
            # 1. CRITICAL: Re-initialize the model using the CURRENT config
            # so the architecture matches the saved checkpoint
            model_to_log = ActivityGatekeeper(best_config_for_run).to(device)
            model_to_log.load_state_dict(torch.load("gatekeeper_best.pth"))
            
            # 3. Log the correctly-configured model
            mlflow.pytorch.log_model(model_to_log, "model", registered_model_name=project_name)
            
            # Set dev alias
            latest_v = utils.client.get_latest_versions(project_name)[0].version
            utils.client.set_registered_model_alias(project_name, "dev", latest_v)
            print(f"Robust Model ({config['note']}) Registered to @dev")
        else:
            print("Mean performance did not beat @dev.")

Initialized MLflow to track repo "SamuelFredricBerg/4dt907"

Repository SamuelFredricBerg/4dt907 initialized!

✅ Random seed set to: 42

--- Fold 1 ---

--- Fold 2 ---

--- Fold 3 ---

--- Fold 4 ---

--- Fold 5 ---

--- Fold 6 ---

--- Fold 7 ---

--- Fold 8 ---

--- Fold 9 ---

--- Fold 10 ---

--- Experiment: GRU_Balanced_Refined ---
Top 5 Worst Performers:
            filename  start_off  stop_off  total_off
48    A88_kinect.csv       88.0      44.0      132.0
109  A133_kinect.csv       45.0      60.0      105.0
90   A138_kinect.csv      100.0       4.0      104.0
19   A116_kinect.csv       41.0      62.0      103.0
117   A53_kinect.csv       48.0      45.0       93.0
Top 5 Best Performers:
            filename  start_off  stop_off  total_off
127  A156_kinect.csv        0.0       1.0        1.0
30    A65_kinect.csv        1.0       1.0        2.0
50    A93_kinect.csv        2.0       0.0        2.0
172   A96_kinect.csv        2.0       0.0        2.0
15    A86_kinect.csv        3.0       0.0        3.0

Final MAE Result -> Total: 13.41 frames, std: 24.27
Start: 15.75 (±17.84) | Stop: 11.06 

2026/05/05 17:15:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/05 17:15:49 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'Start_Stop_Predictor_ModelV2' already exists. Creating a new version of this model...
2026/05/05 17:15:54 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Start_Stop_Predictor_ModelV2, version 4
Created version '4' of model 'Start_Stop_Predictor_ModelV2'.
/var/folders/9w/6516vw7x2gbc3llv8dbzm6vw0000gn/T/ipykernel_56744/105740483.py:188: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https

Robust Model (GRU_Balanced_Refined) Registered to @dev
🏃 View run A12 - new metrics at: https://dagshub.com/SamuelFredricBerg/4dt907.mlflow/#/experiments/0/runs/7c5a9bd2924746abb28ad58fa575bfc1
🧪 View experiment at: https://dagshub.com/SamuelFredricBerg/4dt907.mlflow/#/experiments/0
✅ Random seed set to: 42

--- Fold 1 ---

--- Fold 2 ---

--- Fold 3 ---

--- Fold 4 ---

--- Fold 5 ---

--- Fold 6 ---

--- Fold 7 ---

--- Fold 8 ---

--- Fold 9 ---

--- Fold 10 ---

--- Experiment: LSTM_Dense_Uni ---
Top 5 Worst Performers:
            filename  start_off  stop_off  total_off
48    A88_kinect.csv       98.0      43.0      141.0
134   A78_kinect.csv       74.0      28.0      102.0
109  A133_kinect.csv       44.0      56.0      100.0
57   A121_kinect.csv       73.0      22.0       95.0
90   A138_kinect.csv       92.0       2.0       94.0
Top 5 Best Performers:
            filename  start_off  stop_off  total_off
61     A1_kinect.csv        2.0       1.0        3.0
26   A153_kinect.csv   

# One file check for manual control

In [38]:
import torch
import pandas as pd
import numpy as np

def extract_exercise_bounds(file_path, model_path, config):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ActivityGatekeeper(config).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    df = pd.read_csv(file_path)
    feat_cols = [c for c in df.columns if any(s in c for s in ['_x','_y','_z']) and 'FrameNo' not in c]
    X_data = df[feat_cols].values.astype('float32')
    frame_numbers = df['FrameNo'].values
    
    seq_length = config["seq_length"]
    predictions = []

    with torch.no_grad():
        for i in range(len(df) - seq_length + 1):
            window = X_data[i : i + seq_length]
            window_tensor = torch.tensor(window).unsqueeze(0).to(device)
            
            output = model(window_tensor)
            prob = torch.sigmoid(output).item()
            predictions.append(1 if prob >= 0.5 else 0)

    full_preds = [0] * (seq_length - 1) + predictions
    
    start_frame = None
    stop_frame = None

    for i in range(1, len(full_preds)):
        # Start: 0 -> 1 transition
        if full_preds[i-1] == 0 and full_preds[i] == 1:
            if start_frame is None: start_frame = frame_numbers[i]
        
        # Stop: 1 -> 0 transition
        if full_preds[i-1] == 1 and full_preds[i] == 0:
            stop_frame = frame_numbers[i-1]


    return start_frame, stop_frame

test_csv = "../../data/kinect_good_preprocessed_not_cut_A11_mediapipe/A138_kinect.csv"
model_file = "gatekeeper_best.pth"

start, stop = extract_exercise_bounds(test_csv, model_file, config)

print(f"File: {test_csv}")
print(f"Detected Start Frame: {start}")
print(f"Detected Stop Frame: {stop}")
if start and stop:
    print(f"Total Exercise Duration: {stop - start} frames")
else:
    print("Warning: Model did not detect a clear start/stop transition.")

File: ../../data/kinect_good_preprocessed_not_cut_A11_mediapipe/A138_kinect.csv
Detected Start Frame: 6
Detected Stop Frame: None
